# Notebook to fix PR issues


### Generate the UE data from mobility model first

In [1]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [2]:
import pandas as pd
import scipy
import numpy as np
from radp_library import *
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from radp.digital_twin.mobility.param_regression import get_predicted_alpha,preprocess_ue_data

/Users/tanzimfarhan/Desktop/Maveric/maveric/.venv/lib/python3.11/site-packages/fastkml/config.py:39: UserWarning: Package `lxml` missing. Pretty print will be disabled
  warnings.warn("Package `lxml` missing. Pretty print will be disabled")  # noqa: B028


In [3]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 50,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 10,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 12,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

In [4]:
training_data = get_ue_data(params)
training_data.head()

,mock_ue_id,lon,lat,tick
0,0,48.510339,-16.462645,0
1,1,19.286613,63.617111,0
2,2,21.315702,-47.889952,0
3,3,-70.559364,-79.511709,0
4,4,-168.916011,-39.338640,0


In [28]:
topology = pd.read_csv('/Users/tanzimfarhan/Downloads/topology.csv')
topology.loc[topology['cell_id'] == 'cell_2', 'cell_lat'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lat'] = 90
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lat'] = -90


topology.loc[topology['cell_id'] == 'cell_2', 'cell_lon'] = 0
topology.loc[topology['cell_id'] == 'cell_3', 'cell_lon'] = 180
topology.loc[topology['cell_id'] == 'cell_1', 'cell_lon'] = -180


topology.loc[topology['cell_id'] == 'cell_2', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_3', 'cell_carrier_freq_mhz'] = 2800
topology.loc[topology['cell_id'] == 'cell_1', 'cell_carrier_freq_mhz'] = 2800

In [30]:
def _prepare_all_UEs_from_all_cells_df(
     data, topology
    ) -> pd.DataFrame:
        """
        Connects each user equipment (UE) entry to all cells in the topology for each tick,
        effectively creating a Cartesian product of UEs and cells, which includes data from both sources.
        """

        ue_data = data
        ue_data = ue_data.rename(columns={"lat": "latitude", "lon": "longitude"})
        print("UE Data:", ue_data)
        print("Topology Data:", topology)
        topology_tmp = topology
        # Remove the 'cell_' prefix and convert cell_id to integer if needed
        if topology_tmp["cell_id"].dtype == object:
            topology_tmp["cell_id"] = (
                topology_tmp["cell_id"].str.replace("cell_", "").astype(int)
            )
        ue_data["key"] = 1
        topology_tmp["key"] = 1
        combined_df = pd.merge(ue_data, topology_tmp, on="key").drop("key", axis=1)
        print("Combined DataFrame:", combined_df)
        return combined_df


In [ ]:
def _preprocess_ue_topology_data(data,topology) -> pd.DataFrame:
        full_data = _prepare_all_UEs_from_all_cells_df(data,topology)
        full_data["log_distance"] = full_data.apply(
            lambda row: GISTools.get_log_distance(
                row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
            ),
            axis=1,
        )

        full_data["cell_rxpwr_dbm"] = full_data.apply(
            lambda row: calculate_received_power(
                row["log_distance"], row["cell_carrier_freq_mhz"]
            ),
            axis=1,
        )

        return full_data

In [21]:
def _preprocess_ue_training_data(data,topology) -> pd.DataFrame:
        data = _preprocess_ue_topology_data(data,topology)
        train_per_cell_df = [x for _, x in data.groupby("cell_id")]
        n_cell = len(topology.index)

        metadata_df = pd.DataFrame(
            {
                "cell_id": [cell_id for cell_id in topology.cell_id],
                "idx": [i + 1 for i in range(n_cell)],
            }
        )
        idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))
        desired_idxs = [1 + r for r in range(n_cell)]

        n_samples_train = []
        for df in train_per_cell_df:
            n_samples_train.append(df.shape[0])

        train_per_cell_df_processed = []
        for i in range(n_cell):
            train_per_cell_df_processed.append(
                get_percell_data(
                    data_in=train_per_cell_df[i],
                    choose_strongest_samples_percell=False,
                    n_samples=n_samples_train[i],
                )[0][0]
            )

        training_data = {}

        for i, df in enumerate(train_per_cell_df_processed):
            train_cell_id = idx_cell_id_mapping[i + 1]
            training_data[train_cell_id] = df

        for train_cell_id, training_data_idx in training_data.items():
            training_data_idx["cell_id"] = train_cell_id
            training_data_idx["cell_lat"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_lat"].values[0]
            training_data_idx["cell_lon"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_lon"].values[0]
            training_data_idx["cell_az_deg"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_az_deg"].values[0]
            training_data_idx["cell_carrier_freq_mhz"] = topology[
                topology["cell_id"] == train_cell_id
            ]["cell_carrier_freq_mhz"].values[0]
            training_data_idx["relative_bearing"] = [
                GISTools.get_relative_bearing(
                    training_data_idx["cell_az_deg"].values[0],
                    training_data_idx["cell_lat"].values[0],
                    training_data_idx["cell_lon"].values[0],
                    lat,
                    lon,
                )
                for lat, lon in zip(
                    training_data_idx["latitude"], training_data_idx["longitude"]
                )
            ]

        return training_data

In [22]:
bayesian_digital_twins = {}

In [23]:
def _training(bayesian_digital_twins, maxiter: int, train_data: pd.DataFrame,topology: pd.DataFrame) -> List[float]:
        """
        Trains the Bayesian Digital Twins for each cell in the topology using the UE locations and features
        like log distance, relative bearing, and cell received power (Rx power).
        """
        training_data = _preprocess_ue_training_data(train_data,topology)
        loss_vs_iters = []
        for train_cell_id, training_data_idx in training_data.items():
            bayesian_digital_twins[train_cell_id] = BayesianDigitalTwin(
                data_in=[training_data_idx],
                x_columns=["log_distance", "relative_bearing"],
                y_columns=["cell_rxpwr_dbm"],
                norm_method=NormMethod.MINMAX,
            )
            bayesian_digital_twins[train_cell_id] = bayesian_digital_twins[
                train_cell_id
            ]
            loss_vs_iters.append(
                bayesian_digital_twins[train_cell_id].train_distributed_gpmodel(
                    maxiter=maxiter,
                )
            )
        return bayesian_digital_twins, loss_vs_iters

In [31]:
bayesian_digital_twins, loss_vs_iters = _training(
    bayesian_digital_twins,
    maxiter=100,
    train_data=training_data,
    topology=topology,
)


UE Data:       mock_ue_id   longitude   latitude  tick  key
0              0   48.510339 -16.462645     0    1
1              1   19.286613  63.617111     0    1
2              2   21.315702 -47.889952     0    1
3              3  -70.559364 -79.511709     0    1
4              4 -168.916011 -39.338640     0    1
...          ...         ...        ...   ...  ...
1595          27  -14.856604  -5.730164    49    1
1596          28  -22.307315  40.806648    49    1
1597          29 -179.236707  41.434853    49    1
1598          30   89.100351 -11.651079    49    1
1599          31  146.478589  74.187621    49    1

[1600 rows x 5 columns]
Topology Data:    cell_lat  cell_lon  cell_id  cell_az_deg  cell_carrier_freq_mhz  key
0     -90.0    -180.0        1            0                   2800    1
1       0.0       0.0        2          120                   2800    1
2      90.0     180.0        3          240                   2800    1
Combined DataFrame:       mock_ue_id   longitude   

[2025-05-05 22:22:42,150] INFO:  Iter 1/100 - Loss: 0.768 (delta=inf)
[2025-05-05 22:22:42,180] INFO:  Iter 2/100 - Loss: 0.751 (delta=-0.017659)
[2025-05-05 22:22:42,211] INFO:  Iter 3/100 - Loss: 0.731 (delta=-0.020125)
[2025-05-05 22:22:42,242] INFO:  Iter 4/100 - Loss: 0.716 (delta=-0.014742)
[2025-05-05 22:22:42,272] INFO:  Iter 5/100 - Loss: 0.695 (delta=-0.021033)
[2025-05-05 22:22:42,306] INFO:  Iter 6/100 - Loss: 0.673 (delta=-0.021912)
[2025-05-05 22:22:42,343] INFO:  Iter 7/100 - Loss: 0.657 (delta=-0.015562)
[2025-05-05 22:22:42,375] INFO:  Iter 8/100 - Loss: 0.638 (delta=-0.019893)
[2025-05-05 22:22:42,406] INFO:  Iter 9/100 - Loss: 0.620 (delta=-0.017905)
[2025-05-05 22:22:42,439] INFO:  Iter 10/100 - Loss: 0.599 (delta=-0.020308)
[2025-05-05 22:22:42,468] INFO:  Iter 11/100 - Loss: 0.577 (delta=-0.022224)
[2025-05-05 22:22:42,499] INFO:  Iter 12/100 - Loss: 0.555 (delta=-0.022236)
[2025-05-05 22:22:42,528] INFO:  Iter 13/100 - Loss: 0.534 (delta=-0.021296)
[2025-05-05 22

### Disect all the codes from MRO and put it to radp_library.py
